# Cell-dependent latent shifts for scDisInFact

## Objective

Evaluate whether a cell-dependent latent transition improves prediction of panobinostat responses relative to the original scDisInFact centroid shift. A common backbone is trained once per split and frozen for adapter training. All methods are evaluated against the same held-out target cells.

The primary experiment holds out GBM sample `PW034-705` and evaluates predictions from control cells in patients `PW034` and `PW030`. This notebook implements a single-seed experiment; the extended suite additionally evaluates cell-type holdout and simulated condition-combination generalization.

**Status:** implementation and execution checks are complete; full benchmark results have not been established. No performance improvement is assumed.


In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'cellshift' / 'run.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Repository root not found: expected cellshift/run.py in a parent directory.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from cellshift.data import load_gbm, load_simulated, normalized, split
from cellshift.run import subset, scenarios, metrics, dump
from cellshift.model import Backbone, bandwidth_from_training, train_adapter
from cellshift.upstream import create_scdisinfact_dataset, scdisinfact, REVISION
print('Python executable:', sys.executable)
print('CUDA available:', torch.cuda.is_available())
print('Repository root:', ROOT)


## Data acquisition

The processed GBM dataset is distributed through [Zenodo record 10472715](https://zenodo.org/records/10472715). The downloader verifies the published archive checksum before extraction. Archives are cached under `data/archives`; extracted files are stored under `data`. Existing verified downloads are reused. Concurrent downloads to the same destination are not supported.


In [ ]:
import subprocess

def run_visible(arguments):
    # Use the active kernel interpreter for subprocess execution.
    process = subprocess.Popen([sys.executable, '-u', '-m', *arguments], cwd=ROOT,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait() != 0:
            raise RuntimeError('Subprocess failed; inspect the preceding output.')
    except BaseException:
        process.terminate()
        process.wait()
        raise

run_visible(['cellshift.download', 'gbm'])


## Dataset composition

The input matrix contains observed RNA counts with cells as rows and genes as columns. Metadata identify patient batches, sample identifiers, treatment conditions, and cell types. The following tables summarize the available populations before partitioning.


In [ ]:
data = load_gbm(ROOT / 'data')
print(f'Cells: {data.counts.shape[0]:,}; genes: {data.counts.shape[1]:,}')
display(data.meta[['batch', 'sample_id', 'treatment', 'cell_type']].head(10))
populations = pd.crosstab(data.meta.cell_type, data.meta.treatment)
display(populations)
populations.plot.bar(figsize=(9, 4), title='Number of cells by cell type and treatment')
plt.ylabel('Cells'); plt.tight_layout(); plt.show()


## Training, validation, and test partition

Sample `PW034-705` is reserved exclusively for evaluation. Within the remaining batch × cell-type × condition groups, approximately 20% of cells are assigned to validation. Validation and test cells are excluded from backbone fitting, scaler estimation, and latent-centroid estimation. Validation targets are used only for adapter checkpoint selection.

The split is generated with a fixed random seed and exported with the experiment outputs. This protocol differs from the upstream script, which performs preprocessing before separating the held-out sample.


In [ ]:
SEED = 0
manifest, _ = split(data, 'gbm', SEED, min_cells=50)
display(pd.crosstab(manifest.role, manifest.treatment))
display(manifest.groupby(['role', 'batch', 'cell_type']).size().rename('cells').to_frame())
training = subset(data, manifest.role.eq('train'))
validation = subset(data, manifest.role.eq('val'))
assert not set(training.meta.index) & set(manifest.index[manifest.role.eq('test')])
print('Verified: training and test cell identifiers are disjoint.')


## Experiment configuration

The GBM backbone uses latent dimensions `[8, 2]`, negative-binomial reconstruction, batch size 64, learning rate `5e-4`, and 100 training epochs. The adapter is optimized for at most 100 epochs, with 50 optimization steps per epoch and validation-based early stopping.

CUDA is the default training target. CPU execution requires `ALLOW_CPU=True`. Each configuration execution creates a separate output directory containing the split and experiment parameters.


In [ ]:
from datetime import datetime
import random
ALLOW_CPU = False
BASE_EPOCHS = 100
ADAPTER_EPOCHS = 100
STEPS = 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT = ROOT / 'runs_notebook' / datetime.now().strftime('%Y%m%d_%H%M%S_%f') / 'gbm' / f'seed_{SEED}'
OUT.mkdir(parents=True)
manifest.to_csv(OUT / 'split.csv')
dump(OUT / 'config.json', {'seed': SEED, 'revision': REVISION, 'base_epochs': BASE_EPOCHS,
                          'adapter_epochs': ADAPTER_EPOCHS, 'steps': STEPS, 'device': str(DEVICE)})
print('Training device:', DEVICE)
print('Output directory:', OUT)


## Backbone estimation

Train the upstream scDisInFact implementation on the training partition. The encoder input scaler is fitted on training cells only. After fitting, cache posterior means and compute condition-specific latent centroids from the same partition.

The upstream training loop includes epoch zero; passing `BASE_EPOCHS - 1` yields the specified number of epochs. The diagnostic loss below is computed internally on training-derived subsets and is not an independent validation or test score.


In [ ]:
if DEVICE.type == 'cpu' and not ALLOW_CPU:
    raise RuntimeError('CUDA is unavailable. Set ALLOW_CPU=True to explicitly permit CPU training.')
if (OUT / 'backbone.pt').exists():
    raise RuntimeError('Backbone checkpoint already exists. Use a new output directory for an independent run.')
torch.set_num_threads(4)
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
dictionary = create_scdisinfact_dataset(training.counts, training.meta.copy(),
    condition_key=data.conditions, batch_key='batch', batch_cond_key='sample_id')
dims = [8, 2]
base = scdisinfact(dictionary, Ks=dims, batch_size=64, lr=5e-4, seed=SEED, device=DEVICE)
losses = base.train_model(nepochs=BASE_EPOCHS - 1, recon_loss='NB')
base.eval()
torch.save({'state_dict': base.state_dict(), 'scaler': dictionary['scaler'],
            'matching_dict': dictionary['matching_dict'], 'dims': dims}, OUT / 'backbone.pt')
backbone = Backbone(base, data.conditions)
train_encoded = backbone.encode(training.counts, training.meta)
val_encoded = backbone.encode(validation.counts, validation.meta)
backbone.fit_centroids(train_encoded)
bandwidth = bandwidth_from_training(np.log1p(normalized(training.counts)), SEED)
plt.plot(np.arange(len(losses[0])) * 10, losses[0], marker='o')
plt.xlabel('Epoch'); plt.ylabel('Backbone diagnostic loss'); plt.show()
print('Backbone training complete; checkpoint saved.')


## Baseline evaluation

Evaluate two reference predictors: `fixed`, the original difference of condition-specific latent centroids, and `no_shift`, which leaves biological factors unchanged while applying the destination batch.

Predictions are normalized to a per-cell sum of 100 and aggregated by cell type. MSE, Pearson correlation, and R² compare predicted and observed target-expression centroids. A secondary analysis uses a common denoised reference from the frozen backbone. Groups require at least 50 source and 50 target cells.


In [ ]:
jobs, skipped = scenarios(data, manifest, 'gbm', None, 50)
display(pd.DataFrame(skipped)) if skipped else print('All configured prediction routes meet the evaluation criteria.')

def evaluate_methods(adapters=None):
    rows = []
    for name, smask, tmask, labels, batch in jobs:
        source, target = subset(data, smask), subset(data, tmask)
        enc = backbone.encode(source.counts, source.meta)
        target_enc = backbone.encode(target.counts, target.meta)
        ids = [names.index(value) for names, value in zip(backbone.labels, labels)]
        references = {'raw': normalized(target.counts),
                      'denoised': backbone.predict(target_enc, mode='identity')}
        predictions = {'no_shift': backbone.predict(enc, ids, batch, mode='identity'),
                       'fixed': backbone.predict(enc, ids, batch)}
        for mode, adapter in (adapters or {}).items():
            predictions[mode] = backbone.predict(enc, ids, batch, adapter=adapter)
        for method, prediction in predictions.items():
            for ct in sorted(target.meta.cell_type.unique()):
                si = source.meta.cell_type.eq(ct).values
                ti = target.meta.cell_type.eq(ct).values
                source_mean = normalized(source.counts[si]).mean(0)
                for reference, truth in references.items():
                    rows.append(dict(experiment='gbm', seed=SEED, scenario=name, cell_type=ct,
                                     method=method, reference=reference, n_source=int(si.sum()), n_target=int(ti.sum()),
                                     **metrics(prediction[si], truth[ti], source_mean, bandwidth, SEED)))
    return pd.DataFrame(rows)

def show_results(results):
    observed = results.loc[results.reference.eq('raw')]
    display(observed[['scenario', 'cell_type', 'method', 'mse', 'pearson', 'r2']].round(5))
    observed.pivot_table(index=['scenario', 'cell_type'], columns='method', values='mse').plot.bar(figsize=(12, 5))
    plt.ylabel('MSE against observed test cells (lower is better)')
    plt.tight_layout(); plt.show()


In [ ]:
baseline_results = evaluate_methods()
baseline_results.to_csv(OUT / 'baseline_metrics.csv', index=False)
show_results(baseline_results)


## Frozen-backbone adapter estimation

The cell-dependent model predicts the full latent displacement:

$$\Delta_\phi(z_s,z_u,c_{\mathrm{src}},c_{\mathrm{tgt}}),\qquad z'_u=z_u+\Delta_\phi(\cdot).$$

The adapter is an MLP with two 64-unit ReLU hidden layers. Its inputs are the shared posterior mean, the current condition-specific posterior mean, and one-hot source and target condition labels. The original centroid shift is not added to its output. A cell-independent trainable transition vector (`constant`) controls for improvement attributable solely to fitting a better global shift.

Adapters are trained using multi-bandwidth RBF MMD between predicted expression and frozen-backbone reconstructions of permitted target training cells, after library-size normalization and `log1p`. Transitions remain within the same patient batch and cell type. Cell-type labels group the loss but are not adapter inputs.

The base bandwidth is estimated from at most 512 training cells; relative scales are 0.25, 0.5, 1, 2, and 4. Optimization uses Adam (`lr=1e-3`), at most 128 cells per side, and validation MMD with patience 10. Backbone parameters remain frozen; gradients propagate through the decoder to the adapter. Parameter invariance is checked after training.


In [ ]:
adapters = {}
for mode in ['constant', 'network']:
    print('Training adapter:', mode, flush=True)
    adapter, history = train_adapter(backbone, train_encoded, val_encoded,
        training.meta, validation.meta, dims, mode, bandwidth, SEED,
        epochs=ADAPTER_EPOCHS, steps=STEPS)
    adapters[mode] = adapter
    torch.save({'state_dict': adapter.state_dict(), 'mode': mode, 'dims': dims,
                'categories': [len(x) for x in backbone.labels], 'bandwidth': bandwidth,
                'centroids': [x.cpu() for x in backbone.centroids]}, OUT / f'{mode}.pt')
    frame = pd.DataFrame(history)
    frame.to_csv(OUT / f'{mode}_history.csv', index=False)
    frame.plot(x='epoch', y=['train_mmd', 'val_mmd'], title=mode, figsize=(7, 3))
    plt.ylabel('Distribution matching loss'); plt.show()


## Paired method comparison

Compare `network`, `constant`, `fixed`, and `no_shift` on identical target populations. The primary endpoint is MSE against observed target-expression centroids. Relative improvement is defined as

$$100\,\frac{\mathrm{MSE}_{\mathrm{fixed}}-\mathrm{MSE}_{\mathrm{network}}}{\mathrm{MSE}_{\mathrm{fixed}}}.$$

Positive values indicate lower error than the original fixed shift. Report individual cell-type and prediction-route results, including regressions. A single seed provides descriptive evidence only. Agreement of population means or distributions does not establish accurate responses for individual cells.


In [ ]:
results = evaluate_methods(adapters)
results.to_csv(OUT / 'metrics.csv', index=False)
show_results(results)
paired = results.loc[results.reference.eq('raw')].pivot(index=['scenario', 'cell_type'], columns='method', values='mse')
paired['network_improvement_percent'] = (paired['fixed'] - paired['network']) / paired['fixed'].replace(0, np.nan) * 100
display(paired.round(4))
print('Positive improvement indicates lower MSE than the fixed-shift baseline.')
print('Results directory:', OUT)


## Extended benchmark suite

The extended suite evaluates seeds 0, 1, and 2 for the GBM sample holdout, treated-cell-type holdout, and simulated in-sample and out-of-sample settings. It is controlled by `RUN_EXTENDED` and requires CUDA. Completed command-line runs are reused; incomplete runs require a distinct output directory.

The simulation setting is `2conds_base_10000_500_0.4_100_8`. Out-of-sample evaluation excludes `ctrl + severe` in both batches. Each underlying simulated cell is assigned to a single condition combination to avoid correspondence leakage. Six prediction routes cover single-factor and joint-factor changes with and without batch changes.

**References:** [scDisInFact, Nature Communications (2024)](https://doi.org/10.1038/s41467-024-45227-w); [processed datasets](https://zenodo.org/records/10472715); [pinned upstream implementation](https://github.com/ZhangLabGT/scDisInFact/tree/9466ecf257d923f879cb39712c530d24826d88bc).


In [ ]:
# Enable to execute the multi-seed benchmark suite.
RUN_EXTENDED = False
if RUN_EXTENDED:
    if not torch.cuda.is_available():
        raise RuntimeError('The extended benchmark suite requires CUDA.')
    run_visible(['cellshift.download', 'simulated'])
    for seed in [0, 1, 2]:
        for experiment in ['gbm', 'gbm_celltype', 'sim_is', 'sim_oos']:
            destination = ROOT / 'runs' / experiment / f'seed_{seed}'
            if (destination / 'timing.json').exists() and (destination / 'metrics.csv').exists():
                print('Reusing completed run:', destination)
                continue
            run_visible(['cellshift.run', '--experiment', experiment, '--seed', str(seed), '--device', 'cuda'])
    run_visible(['cellshift.report'])
    from IPython.display import Image
    display(Image(filename=str(ROOT / 'runs' / 'comparison.png')))
    display(Markdown((ROOT / 'runs' / 'REPORT.md').read_text()))
else:
    print('Extended benchmark disabled (RUN_EXTENDED=False).')
